# ECSC Longitudinal Analysis: Y1 vs Y2 vs Y3

Compare context dependency (half-life) across study years using context ablation.

**Dataset**: ECSC transcripts >400 words (126 documents: Y1=60, Y2=35, Y3=31)

**Method**: For each document, measure perplexity on last 40 tokens with varying context lengths, compute half-life (context length to reach 50% of total benefit).

**Hypothesis**: Children may show different memory curve patterns across study years as language development progresses.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Upload Data

Upload your `transcripts.jsonl` file from the ECSC dataset.

In [ ]:
from google.colab import files

print("Upload transcripts.jsonl:")
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_FILE}")

In [ ]:
# Load and filter data
MIN_WORDS = 400

records = []
with open(DATA_FILE) as f:
    for line in f:
        record = json.loads(line)
        record['pop'] = json.loads(record['population'])
        record['word_count'] = len(record['text'].split())
        records.append(record)

df_all = pd.DataFrame(records)
print(f"Total documents: {len(df_all)}")

# Filter by word count
df = df_all[df_all['word_count'] > MIN_WORDS].copy()
df['study_year'] = df['pop'].apply(lambda x: x.get('study_year'))

print(f"Documents with >{MIN_WORDS} words: {len(df)}")
print(f"\nBy study year:")
print(df['study_year'].value_counts().sort_index())

## 2. Load Model

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")

## 3. Context Ablation Functions

In [ ]:
@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """
    Compute perplexity only on tokens in [target_start, target_end].
    Context before target_start is provided but not scored.
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    
    input_ids = torch.tensor([token_ids], device=model.device)
    
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    total_loss = 0.0
    count = 0
    
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        target_token = token_ids[i + 1]
        token_loss = -log_probs[target_token].item()
        total_loss += token_loss
        count += 1
    
    if count == 0:
        return float('inf'), 0
    
    avg_loss = total_loss / count
    perplexity = np.exp(avg_loss)
    
    return perplexity, avg_loss


def analyze_document(text, context_lengths, target_size=40):
    """
    Measure perplexity on the last `target_size` tokens
    with varying amounts of preceding context.
    """
    full_tokens = tokenizer.encode(text)
    n_tokens = len(full_tokens)
    
    # Ensure we have enough tokens
    if n_tokens < target_size + min(context_lengths):
        return []
    
    # Filter context lengths that are achievable
    max_possible_context = n_tokens - target_size
    valid_context_lengths = [c for c in context_lengths if c <= max_possible_context]
    
    results = []
    
    for ctx_len in valid_context_lengths:
        doc_start = n_tokens - target_size - ctx_len
        truncated_tokens = full_tokens[doc_start:]
        
        target_start = len(truncated_tokens) - target_size
        target_end = len(truncated_tokens)
        
        ppl, loss = compute_perplexity_on_region(truncated_tokens, target_start, target_end)
        
        results.append({
            'context_length': ctx_len,
            'perplexity': ppl,
            'loss': loss,
        })
    
    return results


def compute_half_life(contexts, perplexities, percentile=0.5):
    """
    Compute the context length at which `percentile` of total benefit is achieved.
    """
    contexts = np.array(contexts)
    perplexities = np.array(perplexities)
    
    order = np.argsort(contexts)
    contexts = contexts[order]
    perplexities = perplexities[order]
    
    total_benefit = perplexities[0] - perplexities[-1]
    
    if total_benefit <= 0:
        return np.nan
    
    target_ppl = perplexities[0] - percentile * total_benefit
    
    for i in range(len(perplexities) - 1):
        if perplexities[i] >= target_ppl >= perplexities[i+1]:
            frac = (perplexities[i] - target_ppl) / (perplexities[i] - perplexities[i+1])
            return contexts[i] + frac * (contexts[i+1] - contexts[i])
    
    return contexts[-1]

## 4. Run Context Ablation on All Documents

In [ ]:
# Configuration
CONTEXT_LENGTHS = [
    4, 8, 12, 16, 20, 24, 28, 32,
    40, 48, 56, 64,
    80, 96, 112, 128,
    160, 192, 224, 256,
    320, 384
]
TARGET_SIZE = 40

print(f"Context lengths: {len(CONTEXT_LENGTHS)} values from {min(CONTEXT_LENGTHS)} to {max(CONTEXT_LENGTHS)}")
print(f"Target region: last {TARGET_SIZE} tokens")
print(f"Documents to process: {len(df)}")

In [ ]:
# Run analysis
all_results = []
doc_metrics = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing documents"):
    doc_results = analyze_document(row['text'], CONTEXT_LENGTHS, TARGET_SIZE)
    
    if not doc_results:
        continue
    
    # Store raw results
    for r in doc_results:
        r['doc_id'] = row['doc_id']
        r['author_id'] = row['author_id']
        r['study_year'] = row['study_year']
        r['word_count'] = row['word_count']
        all_results.append(r)
    
    # Compute document-level metrics
    contexts = np.array([r['context_length'] for r in doc_results])
    perplexities = np.array([r['perplexity'] for r in doc_results])
    
    half_life = compute_half_life(contexts, perplexities, 0.5)
    total_benefit = perplexities[0] - perplexities[-1] if len(perplexities) > 0 else np.nan
    
    doc_metrics.append({
        'doc_id': row['doc_id'],
        'author_id': row['author_id'],
        'study_year': row['study_year'],
        'word_count': row['word_count'],
        'half_life': half_life,
        'total_benefit': total_benefit,
        'min_perplexity': perplexities[-1] if len(perplexities) > 0 else np.nan,
        'max_perplexity': perplexities[0] if len(perplexities) > 0 else np.nan,
    })

results_df = pd.DataFrame(all_results)
metrics_df = pd.DataFrame(doc_metrics)

print(f"\nCollected {len(results_df)} measurements from {len(metrics_df)} documents")

## 5. Population Comparison: Y1 vs Y2 vs Y3

In [ ]:
def bootstrap_ci(values, n_samples=1000, ci=0.95):
    """Compute bootstrap confidence interval for mean."""
    rng = np.random.default_rng(42)
    boot_means = []
    for _ in range(n_samples):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_means.append(np.mean(sample))
    
    alpha = (1 - ci) / 2
    return np.percentile(boot_means, alpha * 100), np.percentile(boot_means, (1 - alpha) * 100)


print("=" * 70)
print("POPULATION COMPARISON: Study Year")
print("=" * 70)

group_stats = {}
for year in sorted(metrics_df['study_year'].unique()):
    year_df = metrics_df[metrics_df['study_year'] == year]
    half_lives = year_df['half_life'].dropna().values
    
    if len(half_lives) < 2:
        continue
    
    mean_hl = np.mean(half_lives)
    std_hl = np.std(half_lives)
    ci_lower, ci_upper = bootstrap_ci(half_lives)
    
    group_stats[year] = {
        'n': len(half_lives),
        'mean': mean_hl,
        'std': std_hl,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
    }
    
    print(f"\nYear {year}:")
    print(f"  N = {len(half_lives)}")
    print(f"  Half-life: {mean_hl:.1f} +/- {std_hl:.1f} tokens")
    print(f"  95% CI: [{ci_lower:.1f}, {ci_upper:.1f}]")

In [ ]:
# Pairwise statistical comparisons
print("\n" + "=" * 70)
print("PAIRWISE COMPARISONS")
print("=" * 70)

years = sorted(metrics_df['study_year'].unique())
comparisons = []

for i, y1 in enumerate(years):
    for y2 in years[i+1:]:
        vals1 = metrics_df[metrics_df['study_year'] == y1]['half_life'].dropna().values
        vals2 = metrics_df[metrics_df['study_year'] == y2]['half_life'].dropna().values
        
        if len(vals1) < 2 or len(vals2) < 2:
            continue
        
        # Mann-Whitney U test (non-parametric)
        stat, p_value = stats.mannwhitneyu(vals1, vals2, alternative='two-sided')
        
        # Effect size (Cohen's d)
        pooled_std = np.sqrt((np.var(vals1) + np.var(vals2)) / 2)
        cohens_d = (np.mean(vals1) - np.mean(vals2)) / pooled_std if pooled_std > 0 else 0
        
        diff = np.mean(vals1) - np.mean(vals2)
        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
        
        print(f"\nYear {y1} vs Year {y2}:")
        print(f"  Difference: {diff:+.1f} tokens")
        print(f"  Cohen's d: {cohens_d:.3f}")
        print(f"  Mann-Whitney U: {stat:.0f}")
        print(f"  p-value: {p_value:.4f} {sig}")
        
        comparisons.append({
            'group1': y1, 'group2': y2,
            'diff': diff, 'cohens_d': cohens_d,
            'p_value': p_value
        })

In [ ]:
# Kruskal-Wallis test (overall comparison)
print("\n" + "=" * 70)
print("OVERALL TEST (Kruskal-Wallis)")
print("=" * 70)

groups_data = [metrics_df[metrics_df['study_year'] == y]['half_life'].dropna().values 
               for y in years if len(metrics_df[metrics_df['study_year'] == y]) > 0]

if len(groups_data) >= 2:
    h_stat, p_value = stats.kruskal(*groups_data)
    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
    print(f"H-statistic: {h_stat:.2f}")
    print(f"p-value: {p_value:.4f} {sig}")

## 6. Visualizations

In [ ]:
# Color palette for study years
colors = {1: '#3498db', 2: '#e74c3c', 3: '#2ecc71'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top Left: Perplexity curves by study year
ax = axes[0, 0]
for year in sorted(results_df['study_year'].unique()):
    year_df = results_df[results_df['study_year'] == year]
    means = year_df.groupby('context_length')['perplexity'].mean()
    sems = year_df.groupby('context_length')['perplexity'].sem()
    
    ax.errorbar(means.index, means.values, yerr=sems.values,
                marker='o', capsize=3, label=f'Year {year}',
                color=colors.get(year, 'gray'), linewidth=2, markersize=4)

ax.set_xlabel('Context Length (tokens)', fontsize=11)
ax.set_ylabel('Perplexity', fontsize=11)
ax.set_title('Perplexity vs Context by Study Year', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Top Right: Half-life distributions (boxplot)
ax = axes[0, 1]
years_sorted = sorted(metrics_df['study_year'].unique())
half_life_data = [metrics_df[metrics_df['study_year'] == y]['half_life'].dropna().values 
                  for y in years_sorted]
bp = ax.boxplot(half_life_data, labels=[f'Year {y}' for y in years_sorted], patch_artist=True)
for i, (patch, year) in enumerate(zip(bp['boxes'], years_sorted)):
    patch.set_facecolor(colors.get(year, 'gray'))
    patch.set_alpha(0.7)

ax.set_ylabel('Half-Life (tokens)', fontsize=11)
ax.set_title('Half-Life Distribution by Study Year', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Bottom Left: Mean half-life with 95% CI
ax = axes[1, 0]
x_pos = range(len(years_sorted))
means = [group_stats[y]['mean'] for y in years_sorted]
ci_lower = [group_stats[y]['ci_lower'] for y in years_sorted]
ci_upper = [group_stats[y]['ci_upper'] for y in years_sorted]
errors = [[m - l for m, l in zip(means, ci_lower)],
          [u - m for m, u in zip(means, ci_upper)]]

bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
              color=[colors.get(y, 'gray') for y in years_sorted], 
              alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Year {y}\n(n={group_stats[y]["n"]})' for y in years_sorted])
ax.set_ylabel('Mean Half-Life (tokens)', fontsize=11)
ax.set_title('Mean Half-Life with 95% Bootstrap CI', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Bottom Right: Perplexity reduction curves
ax = axes[1, 1]
for year in sorted(results_df['study_year'].unique()):
    year_df = results_df[results_df['study_year'] == year]
    means = year_df.groupby('context_length')['perplexity'].mean().sort_index()
    
    baseline = means.iloc[0]
    reduction = baseline - means
    
    ax.plot(reduction.index, reduction.values,
            marker='o', label=f'Year {year}',
            color=colors.get(year, 'gray'), linewidth=2, markersize=4)

ax.set_xlabel('Context Length (tokens)', fontsize=11)
ax.set_ylabel('Perplexity Reduction', fontsize=11)
ax.set_title('Cumulative Benefit of Context by Study Year', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ecsc_longitudinal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Half-life curves with markers
fig, ax = plt.subplots(figsize=(12, 7))

for year in sorted(results_df['study_year'].unique()):
    year_df = results_df[results_df['study_year'] == year]
    means = year_df.groupby('context_length')['perplexity'].mean()
    sems = year_df.groupby('context_length')['perplexity'].sem()
    
    ax.errorbar(means.index, means.values, yerr=sems.values,
                marker='o', capsize=3, label=f'Year {year}',
                color=colors.get(year, 'gray'), linewidth=2, markersize=5)
    
    # Mark half-life
    hl = group_stats[year]['mean']
    # Interpolate perplexity at half-life
    contexts = means.index.values
    ppls = means.values
    hl_ppl = np.interp(hl, contexts, ppls)
    
    ax.axvline(hl, color=colors.get(year, 'gray'), linestyle='--', alpha=0.5)
    ax.plot(hl, hl_ppl, marker='*', markersize=15, color=colors.get(year, 'gray'),
            markeredgecolor='black', markeredgewidth=1)
    ax.annotate(f'HL={hl:.0f}', (hl, hl_ppl),
                textcoords="offset points", xytext=(10, 10 if year == 1 else -20),
                fontsize=10, color=colors.get(year, 'gray'))

ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Perplexity', fontsize=12)
ax.set_title('Memory Curves by Study Year with Half-Life Markers', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ecsc_memory_curves_with_halflife.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Statistics

In [ ]:
print("\n" + "=" * 70)
print("SUMMARY: Half-Life by Study Year")
print("=" * 70)

summary_data = []
for year in sorted(group_stats.keys()):
    s = group_stats[year]
    summary_data.append({
        'Study Year': year,
        'N': s['n'],
        'Mean HL': f"{s['mean']:.1f}",
        'Std': f"{s['std']:.1f}",
        '95% CI': f"[{s['ci_lower']:.1f}, {s['ci_upper']:.1f}]"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

In [ ]:
# Detailed per-document metrics
print("\n" + "=" * 70)
print("PER-DOCUMENT METRICS (first 20 rows)")
print("=" * 70)
print(metrics_df[['doc_id', 'study_year', 'word_count', 'half_life', 'total_benefit']].head(20).to_string(index=False))

## 8. Save Results

In [ ]:
# Save all results
results_df.to_csv('ecsc_context_ablation_results.csv', index=False)
metrics_df.to_csv('ecsc_document_metrics.csv', index=False)

# Save comparison summary
comparison_summary = {
    'group_stats': {str(k): v for k, v in group_stats.items()},
    'pairwise_comparisons': comparisons,
}
with open('ecsc_population_comparison.json', 'w') as f:
    json.dump(comparison_summary, f, indent=2)

print("Saved:")
print("  - ecsc_context_ablation_results.csv (raw perplexity data)")
print("  - ecsc_document_metrics.csv (per-document half-life)")
print("  - ecsc_population_comparison.json (statistical comparison)")
print("  - ecsc_longitudinal_analysis.png")
print("  - ecsc_memory_curves_with_halflife.png")

In [ ]:
# Download results
from google.colab import files

files.download('ecsc_context_ablation_results.csv')
files.download('ecsc_document_metrics.csv')
files.download('ecsc_population_comparison.json')
files.download('ecsc_longitudinal_analysis.png')
files.download('ecsc_memory_curves_with_halflife.png')

## Interpretation Guide

**Half-Life**: Context length (in tokens) at which 50% of the total perplexity benefit is achieved.

- **Lower half-life** = Most benefit comes from local context (nearby words)
- **Higher half-life** = Benefits from longer-range context (earlier in the text)

**Possible findings:**

- If Y3 > Y2 > Y1: Children's narratives become more coherent over time, benefiting more from longer context
- If Y1 > Y2 > Y3: Younger children's text may be more repetitive/formulaic, needing more context to predict
- If no difference: Context dependency may be stable across development at this age range

**Statistical notes:**
- Mann-Whitney U is non-parametric (doesn't assume normal distribution)
- Cohen's d: |d| < 0.2 = small, 0.2-0.8 = medium, > 0.8 = large effect
- Bootstrap CIs are robust to non-normality